# DS 6021 Final Project: full Ticketmaster pull + data-quality checks

Pulls **every US music event on Ticketmaster** (all 50 states + DC, next 12 months), saves the raw data, then flags junk rows and checks which venues publish prices.

**Before running:** `.env` in this folder with `TM_API_KEY=...`. Run all cells top to bottom.

- **Runtime:** roughly 15–30 minutes. Uses well under the 5,000 calls/day limit, and stops itself at 4,800.
- **Safe to re-run:** finished queries are logged, so if it crashes or you stop it, run again and it picks up where it left off.
- **Weekly re-pulls:** running on a different day creates a new dated folder. Section 7 stacks all pulls so you can track price changes.

**Outputs**
- `data/raw/tm_<date>/events.jsonl`: raw API events, untouched (rubric: keep raw data)
- `data/clean/tm_events_<date>.csv`: one row per event, with quality flags
- `data/clean/tm_events_priced_<date>.csv`: the modeling set (priced, junk removed)

*AI disclosure: drafted with Claude (Anthropic); flag as AI-generated in the repo and describe how it was checked, per the course GenAI policy.*

## 1. Setup

In [1]:
%pip install requests python-dotenv pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(".env")  # .env must sit next to this notebook
TM_KEY = os.getenv("TM_API_KEY")
assert TM_KEY, "TM_API_KEY not found: check that .env is in this folder."

TM_URL = "https://app.ticketmaster.com/discovery/v2/events.json"

STATES = ["AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN",
          "IA","KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH",
          "NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT",
          "VT","VA","WA","WV","WI","WY"]
MONTHS_AHEAD = 12
PAGE_SIZE = 200
PAUSE = 0.25          # seconds between calls (limit is ~5/second)
CALL_BUDGET = 4800    # stop before the 5,000/day limit

PULL_DATE = date.today().isoformat()
RAW_DIR = Path(f"data/raw/tm_{PULL_DATE}")
CLEAN_DIR = Path("data/clean")
RAW_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
EVENTS_FILE = RAW_DIR / "events.jsonl"
DONE_FILE = RAW_DIR / "done_queries.json"

print("Pull date:", PULL_DATE, "| raw folder:", RAW_DIR)

Pull date: 2026-10-19 | raw folder: data\raw\tm_2026-10-19


## 2. Query helpers

Ticketmaster only returns the first ~1,000 results of any search. So each state is searched one month at a time, and any month with more than 1,000 events is split in half (and again if needed) until every piece fits.

In [3]:
calls = 0
done = set(json.loads(DONE_FILE.read_text())) if DONE_FILE.exists() else set()


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")


def tm_get(params):
    """One API call with retries; counts toward the daily budget."""
    global calls
    if calls >= CALL_BUDGET:
        raise RuntimeError("Call budget reached; re-run tomorrow to resume.")
    for attempt in range(4):
        calls += 1
        try:
            r = requests.get(TM_URL, params={**params, "apikey": TM_KEY}, timeout=30)
        except requests.RequestException as e:
            print("  network error, retrying:", e)
            time.sleep(3)
            continue
        time.sleep(PAUSE)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 401:
            raise RuntimeError("401: invalid API key.")
        if r.status_code == 429:
            print("  rate limited, waiting...")
            time.sleep(5 * (attempt + 1))
            continue
        print(f"  HTTP {r.status_code}: {r.text[:120]}")
        return None
    return None


def save_events(events, state):
    with EVENTS_FILE.open("a", encoding="utf-8") as f:
        for ev in events:
            ev["_query_state"] = state
            ev["_pull_date"] = PULL_DATE
            f.write(json.dumps(ev) + "\n")


def mark_done(key):
    done.add(key)
    DONE_FILE.write_text(json.dumps(sorted(done)))


def pull_window(state, start, end):
    """Pull all events for one state between start and end; split if >1,000."""
    key = f"{state}|{iso(start)}|{iso(end)}"
    if key in done:
        return 0
    base = {"classificationName": "music", "countryCode": "US", "stateCode": state,
            "startDateTime": iso(start), "endDateTime": iso(end),
            "size": PAGE_SIZE, "sort": "date,asc"}
    first = tm_get({**base, "page": 0})
    if not first:
        return 0
    total = first.get("page", {}).get("totalElements", 0)
    if total == 0:
        mark_done(key)
        return 0
    if total > 1000 and (end - start) > timedelta(days=1):
        mid = start + (end - start) / 2
        return pull_window(state, start, mid) + pull_window(state, mid, end)

    n = 0
    pages = min(first["page"]["totalPages"], 1000 // PAGE_SIZE)
    for p in range(pages):
        data = first if p == 0 else tm_get({**base, "page": p})
        if not data or "_embedded" not in data:
            break
        evs = data["_embedded"]["events"]
        save_events(evs, state)
        n += len(evs)
    mark_done(key)
    return n

## 3. Run the pull
Prints one line per state. If it stops early (budget, crash, closed laptop), just run this cell again.

In [4]:
now = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
windows = []
for m in range(MONTHS_AHEAD):
    s = now + timedelta(days=30 * m)
    windows.append((s, s + timedelta(days=30)))

t0 = time.time()
for st in STATES:
    n_state = sum(pull_window(st, s, e) for s, e in windows)
    print(f"{st}: {n_state:>5} new events | calls so far: {calls} | {(time.time() - t0) / 60:.1f} min")

print("\nDone. Total calls this session:", calls)

AL:   361 new events | calls so far: 12 | 0.1 min
AK:     4 new events | calls so far: 24 | 0.3 min
AZ:   824 new events | calls so far: 38 | 0.5 min
AR:    60 new events | calls so far: 50 | 0.6 min
CA:  4262 new events | calls so far: 81 | 1.3 min
CO:   925 new events | calls so far: 95 | 1.5 min
CT:   203 new events | calls so far: 107 | 1.6 min
DE:    52 new events | calls so far: 119 | 1.7 min
DC:   379 new events | calls so far: 131 | 1.9 min
FL:  1449 new events | calls so far: 146 | 2.1 min
GA:   828 new events | calls so far: 159 | 2.3 min
HI:    94 new events | calls so far: 171 | 2.5 min
ID:   115 new events | calls so far: 183 | 2.6 min
IL:  1570 new events | calls so far: 199 | 2.9 min
IN:   446 new events | calls so far: 211 | 3.1 min
IA:   188 new events | calls so far: 223 | 3.2 min
KS:    74 new events | calls so far: 235 | 3.4 min
KY:   275 new events | calls so far: 247 | 3.6 min
LA:   383 new events | calls so far: 259 | 3.8 min
ME:    63 new events | calls so far: 

## 4. Flatten raw JSON to a table

In [5]:
def flatten(ev):
    venue = (ev.get("_embedded", {}).get("venues") or [{}])[0]
    cls = (ev.get("classifications") or [{}])[0]
    acts = ev.get("_embedded", {}).get("attractions") or []
    pr = (ev.get("priceRanges") or [{}])[0]
    start = ev.get("dates", {}).get("start", {})
    return {
        "event_id": ev.get("id"),
        "name": ev.get("name"),
        "date": start.get("localDate"),
        "local_time": start.get("localTime"),
        "status": ev.get("dates", {}).get("status", {}).get("code"),
        "onsale_date": ev.get("sales", {}).get("public", {}).get("startDateTime"),
        "venue_id": venue.get("id"),
        "venue": venue.get("name"),
        "city": venue.get("city", {}).get("name"),
        "state": venue.get("state", {}).get("stateCode") or ev.get("_query_state"),
        "postal_code": venue.get("postalCode"),
        "lat": venue.get("location", {}).get("latitude"),
        "lon": venue.get("location", {}).get("longitude"),
        "segment": cls.get("segment", {}).get("name"),
        "genre": cls.get("genre", {}).get("name"),
        "subgenre": cls.get("subGenre", {}).get("name"),
        "headliner": acts[0].get("name") if acts else None,
        "headliner_id": acts[0].get("id") if acts else None,
        "support_acts": "; ".join(a.get("name", "") for a in acts[1:]) or None,
        "n_acts": len(acts),
        "promoter": (ev.get("promoter") or {}).get("name"),
        "ticket_limit": (ev.get("ticketLimit") or {}).get("info"),
        "price_type": pr.get("type"),
        "price_min": pr.get("min"),
        "price_max": pr.get("max"),
        "currency": pr.get("currency"),
        "url": ev.get("url"),
        "pull_date": ev.get("_pull_date"),
    }


with EVENTS_FILE.open(encoding="utf-8") as f:
    raw = [json.loads(line) for line in f]

df = pd.DataFrame(flatten(ev) for ev in raw)
before = len(df)
df = df.drop_duplicates("event_id")   # split windows can overlap at the edges
df["has_price"] = df["price_min"].notna() | df["price_max"].notna()
print(f"{before} rows read, {len(df)} unique events, {df['has_price'].sum()} with a price "
      f"({df['has_price'].mean():.0%})")

31832 rows read, 31681 unique events, 7091 with a price (22%)


## 5. Data-quality flags

Nothing is deleted here. Each problem gets its own flag column so you can show the cleaning steps on a slide and justify each one.

In [6]:
JUNK_WORDS = ["parking", "vip", "package", "upgrade", "add-on", "add on", "gift card",
              "voucher", "meet & greet", "meet and greet", "m&g", "fast lane", "early entry",
              "suite", "lounge", "camping", "shuttle", "hotel", "merch", "pre-show",
              "premium seating", "club access", "experience"]
pattern = "|".join(w.replace("+", r"\+") for w in JUNK_WORDS)

df["flag_junk_name"] = df["name"].str.lower().str.contains(pattern, regex=True, na=False)
df["flag_not_onsale"] = df["status"].isin(["cancelled", "postponed", "rescheduled", "offsale"])
df["flag_zero_price"] = (df["price_min"] == 0) | (df["price_max"] == 0)
df["flag_extreme_price"] = df["price_max"] > 2000
df["flag_min_eq_max"] = df["has_price"] & (df["price_min"] == df["price_max"])
df["flag_non_usd"] = df["has_price"] & df["currency"].notna() & (df["currency"] != "USD")
df["flag_no_headliner"] = df["headliner"].isna()
df["flag_genre_other"] = df["genre"].isin(["Other", "Undefined"]) | df["genre"].isna()

# Same headliner, venue, date and time listed more than once (e.g. GA vs. VIP listings)
dup_cols = ["headliner", "venue_id", "date", "local_time"]
df["flag_duplicate_show"] = df.duplicated(dup_cols, keep="first") & df["headliner"].notna()

flags = [c for c in df.columns if c.startswith("flag_")]
summary = pd.DataFrame({
    "all_events": df[flags].sum(),
    "priced_events": df.loc[df["has_price"], flags].sum(),
})
summary

,all_events,priced_events
flag_junk_name,547,75
flag_not_onsale,1915,514
flag_zero_price,1003,1003
flag_extreme_price,1,1
flag_min_eq_max,4606,4606
flag_non_usd,0,0
flag_no_headliner,5172,4179
flag_genre_other,6540,677
flag_duplicate_show,316,9


In [7]:
# Eyeball each flag: are these really junk? Adjust JUNK_WORDS if good shows got caught.
for flag in ["flag_junk_name", "flag_extreme_price", "flag_min_eq_max", "flag_genre_other"]:
    sample = df.loc[df[flag] & df["has_price"], ["name", "venue", "genre", "price_min", "price_max"]]
    print(f"\n{flag}: {len(sample)} priced rows")
    display(sample.head(8))


flag_junk_name: 75 priced rows


,name,venue,genre,price_min,price_max
1556,SOLD OUT! Josiah Queen: The Damascus Road Expe...,Calvary Community Church,Religious,0.00,0.00
1772,HALLOWEEN SPOOKSHOW – A BENEFIT FOR ROCK AGAI...,Whisky A Go Go,Alternative,48.75,1037.55
2456,"Chicago Tribute Experience, Laurel Canyon Drea...",Garden Amphitheatre,Rock,15.42,15.42
2768,In The End - Linkin Park Experience : Night 1,Garden Amphitheatre,Alternative,23.39,23.39
2769,In The End - Linkin Park Experience : 2 Day Pass,Garden Amphitheatre,Alternative,31.39,31.39
2864,Cory Henry Presents: The Gospel Brunch Experience,Blue Note Los Angeles,Jazz,18.80,55.54
2865,Cory Henry Presents: The Gospel Brunch Experience,Blue Note Los Angeles,Jazz,18.80,55.54
2868,In The End - Linkin Park Experience : Night 2,Garden Amphitheatre,Alternative,23.39,23.39



flag_extreme_price: 1 priced rows


,name,venue,genre,price_min,price_max
4753,Daughtry – 20 Years Unplugged Tour,Ventura Theater,Rock,51.92,2125.82



flag_min_eq_max: 4606 priced rows


,name,venue,genre,price_min,price_max
0,Birmingham Bandstand (Open Mic) at the Nick,The Nick,Rock,4.60,4.60
4,The Filthy Heathens,The Merry Widow,Rock,23.76,23.76
5,Won’t Stay Dead with Turpie & Cursed Coast,The Nick,Rock,8.72,8.72
6,Late Night Songwriters Night at The Nick with ...,The Nick,Alternative,8.72,8.72
11,Badvril at The Nick Wednesday 10/21,The Nick,Alternative,8.72,8.72
14,Molly McGuire + Lirus + Coward God Thursday 10...,The Nick,Rock,8.72,8.72
21,JD SIMO w/ Luther Dickinson X Black Joe Lewis ...,The Merry Widow,Rock,36.64,36.64
22,Dallas Wax,The Fennec - Main Stage,Alternative,18.98,18.98



flag_genre_other: 677 priced rows


,name,venue,genre,price_min,price_max
69,KARLOUS MILLER,Huntsville Levity Live,Other,39.87,52.82
75,KARLOUS MILLER,Huntsville Levity Live,Other,39.87,52.82
86,KARLOUS MILLER,Huntsville Levity Live,Other,39.87,52.82
88,KARLOUS MILLER,Huntsville Levity Live,Other,39.87,52.82
98,KARLOUS MILLER,Huntsville Levity Live,Other,39.87,52.82
207,17th Annual Charlie Brown Jazz Christmas Conce...,Soul Kitchen,Other,38.23,60.87
343,Joe Machi,Huntsville Levity Live,Other,33.39,33.39
345,Joe Machi,Huntsville Levity Live,Other,30.80,30.80


## 6. Which venues publish prices?

If venues are mostly all-priced or all-unpriced, missing prices depend on the venue, not chance. That's a limitation to state in the presentation.

In [8]:
v = (df.groupby(["venue_id", "venue", "state"])
       .agg(events=("event_id", "size"), share_priced=("has_price", "mean"))
       .reset_index())
v_big = v[v["events"] >= 5]

def bucket(s):
    return "all priced (100%)" if s == 1 else "none priced (0%)" if s == 0 else "mixed"

print("Venues with 5+ events:", len(v_big))
display(v_big["share_priced"].map(bucket).value_counts().to_frame("venues"))

print("\nBiggest venues and their priced share:")
display(v_big.sort_values("events", ascending=False).head(20)
        .style.format({"share_priced": "{:.0%}"}))

Venues with 5+ events: 1315


,venues
share_priced,
none priced (0%),1110
all priced (100%),176
mixed,29



Biggest venues and their priced share:


,venue_id,venue,state,events,share_priced
785,KovZpZAF6JEA,Jabbawockeez Theater at MGM Grand Hotel & Casino,NV,532,0%
953,KovZpZAJ6n7A,Saxe Theater at Planet Hollywood Inside the Miracle Mile Mall,NV,341,0%
989,KovZpZAJFnvA,Westgate Las Vegas Cabaret Theater,NV,319,0%
199,KovZ917ASDI,The Rock at La Cantera,TX,283,0%
2612,rZ7HnEZ178J7g,Birdland Jazz Club,NY,276,100%
942,KovZpZAJ666A,V Theater at Planet Hollywood Inside the Miracle Mile Mall,NV,276,0%
2684,rZ7HnEZ17OeJf,Smalls Jazz Club,NY,262,100%
2685,rZ7HnEZ17OePV,Mezzrow,NY,247,100%
144,KovZ917AOSm,Thunderland Showroom at Excalibur,NV,212,0%
3010,rZ7HnEZaASU,Blue Note Jazz Club,NY,206,100%


In [9]:
# Priced share by state (all states, sorted)
(df.groupby("state")["has_price"].agg(events="size", priced="sum", share="mean")
   .sort_values("priced", ascending=False)
   .style.format({"share": "{:.0%}"}))

,events,priced,share
state,,,
NY,3624,1846,51%
CA,4258,1269,30%
IL,1562,657,42%
TN,1051,454,43%
MI,723,322,45%
FL,1428,304,21%
OH,881,287,33%
OR,596,223,37%
LA,379,184,49%


## 7. Save the clean files

In [10]:
df.to_csv(CLEAN_DIR / f"tm_events_{PULL_DATE}.csv", index=False)

drop_flags = ["flag_junk_name", "flag_not_onsale", "flag_zero_price", "flag_non_usd",
              "flag_duplicate_show"]
model_df = df[df["has_price"] & ~df[drop_flags].any(axis=1)].copy()
model_df.to_csv(CLEAN_DIR / f"tm_events_priced_{PULL_DATE}.csv", index=False)

print(f"All events:       {len(df):>6}")
print(f"Priced:           {df['has_price'].sum():>6}")
print(f"Modeling set:     {len(model_df):>6}  (priced, junk removed)")
print(f"  DMV (DC/MD/VA): {model_df['state'].isin(['DC', 'MD', 'VA']).sum():>6}")
print("\nKept but flagged for a decision: extreme prices, min = max, missing headliner, genre 'Other'.")
model_df[["price_min", "price_max"]].describe().round(2)

All events:        31681
Priced:             7091
Modeling set:       5613  (priced, junk removed)
  DMV (DC/MD/VA):     46

Kept but flagged for a decision: extreme prices, min = max, missing headliner, genre 'Other'.


,price_min,price_max
count,5613.00,5613.00
mean,31.93,65.32
std,18.55,112.65
min,0.26,0.26
25%,21.43,24.61
50%,28.75,39.10
75%,39.29,56.06
max,395.54,2125.82


## 8. Stack weekly pulls (run after 2+ pulls)

Each pull folder is a snapshot. Stacking them shows how an event's listed price changes as the show gets closer.

In [11]:
pull_files = sorted(CLEAN_DIR.glob("tm_events_2*.csv"))
print("Pulls found:", [p.name for p in pull_files])
if len(pull_files) >= 2:
    panel = pd.concat(pd.read_csv(p) for p in pull_files)
    panel = panel[panel["has_price"]]
    changes = (panel.groupby("event_id")["price_max"].agg(["nunique", "min", "max"])
                    .query("nunique > 1"))
    print(f"Events seen in 2+ pulls: {panel['event_id'].duplicated().sum()}")
    print(f"Events whose max price changed: {len(changes)}")
    display(changes.head(10))
else:
    print("Only one pull so far; run this notebook again next week.")

Pulls found: ['tm_events_2026-10-19.csv']
Only one pull so far; run this notebook again next week.
